Cell 1 — Setup

In [0]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from dateutil.relativedelta import relativedelta

INITIAL_CAPITAL = 5_000_000
TRADING_DAYS = 8760         # 365*24 - hourly bars, continous
COST = 0.0005

# ── MMF Configuration ──────────────────────────────────────
MMF_ANNUAL_RATE  = 0.052                              # fallback for pre-PMMF bars
MMF_HOURLY_RATE  = (1 + MMF_ANNUAL_RATE) ** (1/8760) - 1


Cell 2 — Load Signal Data (Features + Regime)

In [0]:
CATALOG = "pawan_quant_research_1"
SCHEMA = "default"

SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_bt_master_grid_1h"
LABEL_PATH = "/Volumes/pawan_quant_research_1/v4/market_data/bt_fred_1d_regime_signals-v4.csv"

FEATURES = [
    "CPIAUCSL_z","PCEPI_z","PCE_z","PAYEMS_z","UNRATE_z",
    "GDPC1_z","INDPRO_z","UMCSENT_z","VIXCLS_z","vix3m_z",
    "VIX_Inversion_Z","Price_Mom_24H","CPI_Event_Active","NFP_Event_Active"
]

# ─────────────────────────────────────────────
# LOAD SILVER (HOURLY FEATURES)
# ─────────────────────────────────────────────
spark_df = spark.table(SILVER_TABLE).select(["timestamp"] + FEATURES)
silver_df = spark_df.toPandas()
silver_df["timestamp"] = pd.to_datetime(silver_df["timestamp"])


# ─────────────────────────────────────────────
# LOAD LABELS (DAILY → WILL BE ALIGNED)
# ─────────────────────────────────────────────
labels_df = pd.read_csv(LABEL_PATH, parse_dates=["timestamp"])
labels_df = labels_df[["timestamp","predicted_regime"]]
labels_df = labels_df.rename(columns={"predicted_regime":"regime"})


# ─────────────────────────────────────────────
# ALIGN LABELS TO HOURLY GRID (KEY FIX)
# ─────────────────────────────────────────────
df = silver_df.merge(labels_df, on="timestamp", how="left")

# Forward-fill regime so hourly data is fully covered
df["regime"] = df["regime"].ffill()

# Drop only initial missing values
df = df.dropna(subset=["regime"])

df["regime"] = df["regime"].astype(int)


# ─────────────────────────────────────────────
# SANITY CHECK (MANDATORY)
# ─────────────────────────────────────────────
print("DF RANGE:", df["timestamp"].min(), "→", df["timestamp"].max())
print("Rows:", len(df))

In [0]:
print("Label range:", labels_df.timestamp.min(), "→", labels_df.timestamp.max())

Cell 3 — Walk-Forward Schedule

In [0]:
TRAIN_START = "2016-01-01"
LIVE_START  = "2021-01-01"
LIVE_END    = "2026-01-01"

train_start = pd.Timestamp(TRAIN_START)
live_start  = pd.Timestamp(LIVE_START)
live_end    = pd.Timestamp(LIVE_END)

retrain_dates = []
d = live_start

while d <= live_end:
    retrain_dates.append(d)
    d += relativedelta(months=1)

In [0]:
print("WF RANGE:", wf_df.index.min(), "→", wf_df.index.max())

Cell 4 — Train Function

In [0]:
def train_model(train_df):

    model = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softprob",   # IMPORTANT CHANGE
        num_class=3,
        tree_method="hist"
    )

    model.fit(train_df[FEATURES], train_df["regime"])

    return model

Cell 5 — WalkForward + Probabilities (CORE CHANGE)

In [0]:
# ============================================================
# WALK-FORWARD — FIXED (ENSURES FULL COVERAGE)
# ============================================================

wf_data = []

for i in range(len(retrain_dates) - 1):

    train_end = retrain_dates[i]
    test_end  = retrain_dates[i + 1]

    train_df = df[(df.timestamp >= train_start) & (df.timestamp < train_end)]
    test_df  = df[(df.timestamp >= train_end) & (df.timestamp < test_end)].copy()

    # --- FIX: skip only if training data is insufficient ---
    if len(train_df) < 500:
        continue

    # --- FIX: allow small test windows (do NOT skip) ---
    if len(test_df) == 0:
        continue

    model = train_model(train_df)

    probs = model.predict_proba(test_df[FEATURES])
    preds = np.argmax(probs, axis=1)

    test_df["prob_0"] = probs[:, 0]
    test_df["prob_1"] = probs[:, 1]
    test_df["prob_2"] = probs[:, 2]
    test_df["predicted_regime"] = preds

    wf_data.append(test_df)


# ============================================================
# COMBINE (CRITICAL FIX)
# ============================================================

wf_df = pd.concat(wf_data)
wf_df = wf_df.sort_values("timestamp")
wf_df = wf_df.drop_duplicates(subset=["timestamp"])
wf_df = wf_df.set_index("timestamp")


# ============================================================
# SANITY CHECK
# ============================================================

print("WF RANGE:", wf_df.index.min(), "→", wf_df.index.max())
print("WF rows:", len(wf_df))
print("Regime distribution:")
print(wf_df["predicted_regime"].value_counts(normalize=True))

Cell 6 — Combine WF Data

In [0]:
wf_df = pd.concat(wf_data).sort_values("timestamp")
wf_df = wf_df.set_index("timestamp")

Cell 7 — Load Prices

In [0]:
# Cell 7 — Load Prices (SPY, QQQ, PMMF)

prices = spark.table(
    "pawan_quant_research_1.v4.bronze_bt_market_1h"
).filter("symbol IN ('SPY', 'QQQ', 'PMMF')").toPandas()

prices["timestamp"] = pd.to_datetime(prices["timestamp"])

prices = prices.pivot_table(
    index="timestamp",
    columns="symbol",
    values="close",
    aggfunc="last"
)

# ── Drop existing SPY/QQQ columns from wf_df before joining ──
# They came from the silver feature grid and are not needed as features
cols_to_drop = [c for c in ["SPY", "QQQ", "PMMF", "SPY_close", "QQQ_close"] if c in wf_df.columns]
if cols_to_drop:
    wf_df = wf_df.drop(columns=cols_to_drop)
    print(f"Dropped existing price columns: {cols_to_drop}")

# ── Join clean prices ──
wf_df = wf_df.join(prices)

wf_df["SPY"]  = wf_df["SPY"].ffill()
wf_df["QQQ"]  = wf_df["QQQ"].ffill()
wf_df["PMMF"] = wf_df["PMMF"].ffill()

# Only require SPY/QQQ — SGOV may have gaps in early years (2016–2020)
wf_df = wf_df.dropna(subset=["SPY", "QQQ"])

print(f"wf_df shape after price join: {wf_df.shape}")
print(f"PMMF nulls remaining: {wf_df['PMMF'].isna().sum()}")
print(wf_df[["SPY", "QQQ", "PMMF"]].tail(3))


Cell 8 — Baseline (WF Hard Allocation)

In [0]:
def run_wf_baseline(df):

    capital = INITIAL_CAPITAL
    portfolio = [capital]
    active = "START"

    for i in range(1, len(df)):

        r = df.predicted_regime.iloc[i-1]

        if r == 0:
            target = "QQQ"
            ret = df.QQQ.iloc[i] / df.QQQ.iloc[i-1] - 1

        elif r == 1:
            target = "SPY"
            ret = df.SPY.iloc[i] / df.SPY.iloc[i-1] - 1

        else:
            target = "PMMF"
            p_now  = df.PMMF.iloc[i]
            p_prev = df.PMMF.iloc[i-1]
            if pd.notna(p_now) and pd.notna(p_prev) and p_prev > 0:
                ret = p_now / p_prev - 1
            else:
                ret = MMF_HOURLY_RATE          # accrual fallback

        cost = COST if target != active else 0
        capital *= (1 + ret - cost)

        portfolio.append(capital)
        active = target

    return portfolio

wf_df["WF_Baseline"] = run_wf_baseline(wf_df)


Cell 9 — Conviction Allocation (WF)

In [0]:
CONVICTION_THRESHOLD = 0.30

def run_wf_prob(df):

    capital = INITIAL_CAPITAL
    portfolio = [capital]

    for i in range(1, len(df)):

        w0 = df.prob_0.iloc[i-1]
        w1 = df.prob_1.iloc[i-1]
        w2 = df.prob_2.iloc[i-1]

        r_qqq = df.QQQ.iloc[i] / df.QQQ.iloc[i-1] - 1
        r_spy = df.SPY.iloc[i] / df.SPY.iloc[i-1] - 1

        # PMMF return — real price if available, else accrual
        p_now  = df.PMMF.iloc[i]
        p_prev = df.PMMF.iloc[i-1]
        if pd.notna(p_now) and pd.notna(p_prev) and p_prev > 0:
            r_mmf = p_now / p_prev - 1
        else:
            r_mmf = MMF_HOURLY_RATE

        max_prob = max(w0, w1, w2)

        # HIGH CONVICTION: hard-switch
        if max_prob >= CONVICTION_THRESHOLD:
            if w0 == max_prob:
                port_ret = r_qqq
            elif w1 == max_prob:
                port_ret = r_spy
            else:
                port_ret = r_mmf              # PMMF instead of 0

        # LOW CONVICTION: soft probability blend
        else:
            port_ret = w0 * r_qqq + w1 * r_spy + w2 * r_mmf   # PMMF replaces 0

        # Turnover cost
        if i == 1:
            turnover = 1
        else:
            turnover = (
                abs(df.prob_0.iloc[i-1] - df.prob_0.iloc[i-2]) +
                abs(df.prob_1.iloc[i-1] - df.prob_1.iloc[i-2]) +
                abs(df.prob_2.iloc[i-1] - df.prob_2.iloc[i-2])
            ) / 2

        capital *= (1 + port_ret - COST * turnover)
        portfolio.append(capital)

    return portfolio

wf_df["WF_Prob"] = run_wf_prob(wf_df)


Cell 10 — SPY Benchmark

In [0]:
wf_df["SPY_Value"] = (
    wf_df["SPY"] / wf_df["SPY"].iloc[0]
) * INITIAL_CAPITAL

Cell 11 — Metrics

In [0]:
def compute_metrics(series):
    r = series.pct_change().dropna()

    years = len(r) / TRADING_DAYS

    total_return = series.iloc[-1] / series.iloc[0] - 1
    cagr = (series.iloc[-1] / series.iloc[0]) ** (1 / years) - 1
    vol = r.std() * np.sqrt(TRADING_DAYS)
    sharpe = r.mean() / r.std() * np.sqrt(TRADING_DAYS)

    dd = series / series.cummax() - 1
    maxdd = dd.min()

    calmar = cagr / abs(maxdd) if maxdd != 0 else np.nan   # NEW

    return {
        "Final Value":   series.iloc[-1],
        "Total Return":  total_return,
        "CAGR":          cagr,
        "Volatility":    vol,
        "Sharpe":        sharpe,
        "Max Drawdown":  maxdd,
        "Calmar":        calmar,            # NEW
    }

def min_balance_vs_initial(series):
    initial = series.iloc[0]
    min_val = series.min()
    abs_dd_initial = min_val / initial - 1   # e.g. -0.40 = -40% vs start
    return min_val, abs_dd_initial


metrics = pd.DataFrame([
    compute_metrics(wf_df["WF_Baseline"]),
    compute_metrics(wf_df["WF_Prob"]),
    compute_metrics(wf_df["SPY_Value"]),
], index=["V4 WalkForward", "V4 WF + Prob", "SPY"])

metrics_pretty = metrics.copy()
metrics_pretty["Final Value"] = metrics_pretty["Final Value"].map(lambda x: f"{x:,.0f}")

for col in ["Total Return", "CAGR", "Volatility", "Max Drawdown"]:
    metrics_pretty[col] = metrics_pretty[col].map(lambda x: f"{x*100:,.2f}%")

metrics_pretty["Sharpe"] = metrics_pretty["Sharpe"].map(lambda x: f"{x:.2f}")
metrics_pretty["Calmar"] = metrics_pretty["Calmar"].map(lambda x: f"{x:.2f}")  # NEW

metrics_pretty.index.name = "Strategy"
display(metrics_pretty.reset_index())


Cell 11b - Daily Hit Rate Analysis

In [0]:
# Cell 11b — Daily Hit Rate Analysis

# Step 1: Resample hourly equity → daily NAV
# Cell 11b — Daily Hit Rate Analysis (Fixed)

daily_nav = pd.DataFrame({
    "V4 WalkForward": wf_df["WF_Baseline"],
    "V4 WF + Prob":   wf_df["WF_Prob"],
    "SPY":            wf_df["SPY_Value"],
}).resample("D").last().dropna()

daily_ret = daily_nav.pct_change().dropna()

# ── Threshold buckets ──────────────────────────────────────────
# >= 0       : strict (original)
# >= -0.001  : near-flat days (±10bps) counted as neutral, not loss
# > 0        : strictly positive only
thresholds = {
    "Hit Rate (>=0, strict)":       daily_ret >= 0,
    "Hit Rate (>= -10bps, lenient)": daily_ret >= -0.001,
    "Hit Rate (strictly > 0)":      daily_ret > 0,
}

rows = []
for label, mask in thresholds.items():
    row = (mask.sum() / len(daily_ret) * 100).round(2)
    row.name = label
    rows.append(row)

hr_table = pd.DataFrame(rows)
hr_table.columns.name = "Strategy"
display(hr_table.applymap(lambda x: f"{x:.2f}%"))

# ── Yearly breakdown (strict >= 0) ────────────────────────────
yearly_hit = (
    daily_ret >= 0
).resample("YE").apply(lambda x: x.sum() / len(x))
yearly_hit.index = yearly_hit.index.year

yearly_fmt = yearly_hit.applymap(lambda x: f"{x*100:.1f}%")
display(yearly_fmt.reset_index().rename(columns={"index": "Year"}))

# ── Median return per strategy (key diagnostic) ───────────────
print("\n--- Median Daily Return ---")
print(daily_ret.median().map(lambda x: f"{x*100:.4f}%"))

# ── Leverage eligibility on STRICT threshold ─────────────────
print("\n--- Leverage Eligibility (strict >= 0, > 60%) ---")
strict_hr = (daily_ret >= 0).sum() / len(daily_ret)
for strat, hr in strict_hr.items():
    flag = "✅ YES" if hr > 0.60 else "❌ NO"
    print(f"  {strat}: {hr*100:.2f}%  {flag}")


In [0]:
# Cell 11c — Diagnose WF_Prob Hit Rate Bug

# Check 1: Are there NaNs or zeros in WF_Prob?
print("WF_Prob nulls:", wf_df["WF_Prob"].isna().sum())
print("WF_Prob zeros:", (wf_df["WF_Prob"] == 0).sum())
print("WF_Prob min value:", wf_df["WF_Prob"].min())

# Check 2: Are there large jumps (>10% single hour) in WF_Prob?
prob_hourly_ret = wf_df["WF_Prob"].pct_change().dropna()
large_moves = prob_hourly_ret[prob_hourly_ret.abs() > 0.10]
print(f"\nHourly returns > 10%: {len(large_moves)}")
print(large_moves.head(10))

# Check 3: Compare daily return distributions side by side
daily_nav = pd.DataFrame({
    "V4 WalkForward": wf_df["WF_Baseline"],
    "V4 WF + Prob":   wf_df["WF_Prob"],
    "SPY":            wf_df["SPY_Value"],
}).resample("D").last().dropna()

daily_ret = daily_nav.pct_change().dropna()

print("\n--- Daily Return Stats ---")
print(daily_ret.describe().round(4))

# Check 4: How many days have EXTREME negative returns in WF_Prob?
print("\nWF+Prob days with return < -5%:", (daily_ret["V4 WF + Prob"] < -0.05).sum())
print("WF+Prob days with return > +5%:", (daily_ret["V4 WF + Prob"] > 0.05).sum())
print("WF Baseline days with return < -5%:", (daily_ret["V4 WalkForward"] < -0.05).sum())


Cell 12 — Plot

In [0]:
plt.figure(figsize=(16,7))

plt.plot(wf_df["WF_Baseline"], label="V4 WalkForward")
plt.plot(wf_df["WF_Prob"], label="V4 WF + Prob")
plt.plot(wf_df["SPY_Value"], label="SPY")

plt.yscale("log")
plt.legend()
plt.grid(alpha=0.3)
plt.title("WF Baseline vs Probabilistic Allocation")
plt.show()

Cell 13 - Leverage Strategy

In [0]:
# Cell 13 — V4 WalkForward + DD-Scaled Leverage (Hard 30% Cap)

BASE_LEVERAGE      = 2.0        # max leverage at low DD
MIN_LEVERAGE       = 1.0       # no leverage below this
HIT_THRESHOLD      = 0.60       # rolling hit-rate gate
LEVERAGE_LOOKBACK  = 24 * 20    # 20-day window (hourly)
DD_HARD_CAP        = -0.30      # do NOT allow DD worse than this
DD_FULL_LEV        = -0.05      # DD >= -5% → full 2.0x
DD_NO_LEV          = -0.25      # DD <= -25% → min 1.0x (no leverage)

def run_wf_baseline_levered_dd_scaled(df):

    capital     = INITIAL_CAPITAL
    portfolio   = [capital]
    active      = "START"
    hourly_rets = []
    peak        = capital

    for i in range(1, len(df)):

        r = df.predicted_regime.iloc[i-1]

        if r == 0:
            target = "QQQ"
            ret    = df.QQQ.iloc[i] / df.QQQ.iloc[i-1] - 1

        elif r == 1:
            target = "SPY"
            ret    = df.SPY.iloc[i] / df.SPY.iloc[i-1] - 1

        else:
            target = "PMMF"
            p_now  = df.PMMF.iloc[i]
            p_prev = df.PMMF.iloc[i-1]
            if pd.notna(p_now) and pd.notna(p_prev) and p_prev > 0:
                ret = p_now / p_prev - 1
            else:
                ret = MMF_HOURLY_RATE

        # --- update peak & current drawdown ---
        if capital > peak:
            peak = capital
        dd = capital / peak - 1.0   # negative in drawdown

        # --- rolling hit-rate gate ---
        if len(hourly_rets) >= LEVERAGE_LOOKBACK:
            window   = hourly_rets[-LEVERAGE_LOOKBACK:]
            hit_rate = sum(r2 >= -0.001 for r2 in window) / len(window)
            lev_hr   = BASE_LEVERAGE if hit_rate > HIT_THRESHOLD else 1.0
        else:
            lev_hr = 1.0

        # --- DD–scaled leverage factor ---
        if dd >= DD_FULL_LEV:
            lev_dd = BASE_LEVERAGE                     # shallow DD → full lev
        elif dd <= DD_NO_LEV:
            lev_dd = MIN_LEVERAGE                      # deep DD → no lev
        else:
            # linear scale between DD_FULL_LEV and DD_NO_LEV
            frac   = (dd - DD_NO_LEV) / (DD_FULL_LEV - DD_NO_LEV)
            lev_dd = MIN_LEVERAGE + (BASE_LEVERAGE - MIN_LEVERAGE) * frac

        lev = min(lev_hr, lev_dd)

        # never lever PMMF (cash)
        if target == "PMMF":
            lev = 1.0

        # --- apply hard 30% DD cap safeguard ---
        if dd <= DD_HARD_CAP and lev > 1.0:
            lev = 1.0

        cost = COST if target != active else 0
        capital *= (1 + ret * lev - cost)

        portfolio.append(capital)
        hourly_rets.append(ret)
        active = target

    return portfolio

wf_df["WF_Baseline_Lev"] = run_wf_baseline_levered_dd_scaled(wf_df)
print("✅ Leveraged strategy with DD-scaled cap computed")


Cell 14 - Leveraged Metrics

In [0]:
# Cell 14 — Metrics: Baseline vs Leveraged vs SPY

lev_metrics = pd.DataFrame([
    compute_metrics(wf_df["WF_Baseline"]),
    compute_metrics(wf_df["WF_Baseline_Lev"]),
    compute_metrics(wf_df["SPY_Value"]),
], index=["V4 WalkForward", "V4 WalkForward + Lev", "SPY"])

lev_pretty = lev_metrics.copy()
lev_pretty["Final Value"]  = lev_pretty["Final Value"].map(lambda x: f"{x:,.0f}")
lev_pretty["Calmar"]       = lev_pretty["Calmar"].map(lambda x: f"{x:.2f}")
lev_pretty["Sharpe"]       = lev_pretty["Sharpe"].map(lambda x: f"{x:.2f}")

for col in ["Total Return", "CAGR", "Volatility", "Max Drawdown"]:
    lev_pretty[col] = lev_pretty[col].map(lambda x: f"{x*100:.2f}%")

lev_pretty.index.name = "Strategy"
display(lev_pretty.reset_index())


Cell 15 - Hit Rate for Leveraged Strategy

In [0]:
# Cell 15 — Hit Rate: Baseline vs Leveraged vs SPY

HIT_THRESHOLD_DISPLAY = -0.001

lev_daily_nav = pd.DataFrame({
    "V4 WalkForward":       wf_df["WF_Baseline"],
    "V4 WalkForward + Lev": wf_df["WF_Baseline_Lev"],
    "SPY":                  wf_df["SPY_Value"],
}).resample("D").last().dropna()

lev_daily_ret = lev_daily_nav.pct_change().dropna()

# Overall hit rate
hit_rate = (lev_daily_ret >= HIT_THRESHOLD_DISPLAY).sum() / len(lev_daily_ret)

hr_lev = pd.DataFrame({
    "Win Days":   (lev_daily_ret >= HIT_THRESHOLD_DISPLAY).sum(),
    "Total Days": len(lev_daily_ret),
    "Hit Rate":   hit_rate.map(lambda x: f"{x*100:.2f}%"),
    "Lev Eligible (>60%)": hit_rate.map(
        lambda x: "✅ YES" if x > 0.60 else "❌ NO"
    ),
})
hr_lev.index.name = "Strategy"
display(hr_lev.reset_index())

# Yearly breakdown
yearly_lev = (
    lev_daily_ret >= HIT_THRESHOLD_DISPLAY
).resample("YE").apply(lambda x: x.sum() / len(x))
yearly_lev.index = yearly_lev.index.year

print("\n--- Yearly Hit Rate ---")
display(yearly_lev.map(lambda x: f"{x*100:.1f}%")
        .reset_index().rename(columns={"index": "Year"}))


Cell 16 - Plot Baseline vs Leveraged vs SPY

In [0]:
# Cell 16 — Equity Curve: Baseline vs Leveraged vs SPY

plt.figure(figsize=(16, 7))

plt.plot(wf_df.index, wf_df["WF_Baseline"],
         label="V4 WalkForward",          linewidth=1.5)
plt.plot(wf_df.index, wf_df["WF_Baseline_Lev"],
         label="V4 WalkForward + Lev",    linewidth=1.5, linestyle="--")
plt.plot(wf_df.index, wf_df["SPY_Value"],
         label="SPY",                     linewidth=1.5)

plt.yscale("log")
plt.legend()
plt.grid(alpha=0.3)
plt.title("V4 WalkForward vs Leveraged vs SPY")
plt.tight_layout()
plt.show()


In [0]:
# Cell 13 — V4 WalkForward + 5x Leverage, DD vs Initial ≤ 50%

BASE_LEVERAGE      = 5.0       # max gross leverage when conditions are ideal
MIN_LEVERAGE       = 1.0       # no leverage below this
HIT_THRESHOLD      = 0.60      # rolling hit-rate gate
LEVERAGE_LOOKBACK  = 24 * 20   # 20-day window (hourly)

ABS_DD_HARD_CAP    = -0.50     # vs INITIAL_CAPITAL, not peak
DD_FULL_LEV        = -0.05     # DD >= -5% → full 5x
DD_NO_LEV          = -0.35     # DD <= -35% → min 1x (no leverage)


def run_wf_baseline_levered_5x(df):

    capital     = INITIAL_CAPITAL
    portfolio   = [capital]
    active      = "START"
    hourly_rets = []
    peak        = capital      # still track peak for info, but cap uses initial

    for i in range(1, len(df)):

        r = df.predicted_regime.iloc[i-1]

        # --- regime returns (unchanged) ---
        if r == 0:
            target = "QQQ"
            ret    = df.QQQ.iloc[i] / df.QQQ.iloc[i-1] - 1

        elif r == 1:
            target = "SPY"
            ret    = df.SPY.iloc[i] / df.SPY.iloc[i-1] - 1

        else:
            target = "PMMF"
            p_now  = df.PMMF.iloc[i]
            p_prev = df.PMMF.iloc[i-1]
            if pd.notna(p_now) and pd.notna(p_prev) and p_prev > 0:
                ret = p_now / p_prev - 1
            else:
                ret = MMF_HOURLY_RATE

        # --- update peak (for info) ---
        if capital > peak:
            peak = capital

        # DD vs peak (diagnostic)
        dd_peak = capital / peak - 1.0

        # DD vs INITIALCAPITAL — this is the constraint we care about
        dd_initial = capital / INITIAL_CAPITAL - 1.0   # negative once < initial

        # --- rolling hit-rate gate (same as 2x version) ---
        if len(hourly_rets) >= LEVERAGE_LOOKBACK:
            window   = hourly_rets[-LEVERAGE_LOOKBACK:]
            hit_rate = sum(r2 >= -0.001 for r2 in window) / len(window)
            lev_hr   = BASE_LEVERAGE if hit_rate > HIT_THRESHOLD else 1.0
        else:
            lev_hr = 1.0

        # --- DD–scaled leverage factor based on dd_initial ---
        # dd_initial in [0, -35%] → scale 5x → 1x linearly
        if dd_initial >= DD_FULL_LEV:
            lev_dd = BASE_LEVERAGE
        elif dd_initial <= DD_NO_LEV:
            lev_dd = MIN_LEVERAGE
        else:
            frac   = (dd_initial - DD_NO_LEV) / (DD_FULL_LEV - DD_NO_LEV)
            lev_dd = MIN_LEVERAGE + (BASE_LEVERAGE - MIN_LEVERAGE) * frac

        lev = min(lev_hr, lev_dd)

        # never lever PMMF (cash)
        if target == "PMMF":
            lev = 1.0

        # --- absolute 50% DD cap vs initial capital ---
        # if equity <= 50% of initial, force no leverage (>1x)
        if dd_initial <= ABS_DD_HARD_CAP and lev > 1.0:
            lev = 1.0

        cost = COST if target != active else 0
        capital *= (1 + ret * lev - cost)

        portfolio.append(capital)
        hourly_rets.append(ret)
        active = target

    return portfolio


wf_df["WF_Baseline_Lev"] = run_wf_baseline_levered_5x(wf_df)
print("✅ 5x leverage strategy with absolute 50% DD cap computed")


In [0]:
# Cell 14 — Metrics: Baseline vs Leveraged vs SPY
# adds Min Balance and Abs DD vs Initial

def min_balance_vs_initial(series):
    initial = series.iloc[0]
    min_val = series.min()
    abs_dd_initial = min_val / initial - 1.0   # e.g. -0.40 = -40% vs start
    return min_val, abs_dd_initial

rows = []
index = []

for name, s in [
    ("V4 WalkForward",       wf_df["WF_Baseline"]),
    ("V4 WalkForward + Lev", wf_df["WF_Baseline_Lev"]),
    ("SPY",                  wf_df["SPY_Value"]),
]:
    m = compute_metrics(s)
    min_bal, abs_dd_init = min_balance_vs_initial(s)
    m["Min Balance"]       = min_bal
    m["Abs DD vs Initial"] = abs_dd_init
    rows.append(m)
    index.append(name)

lev_metrics = pd.DataFrame(rows, index=index)

lev_pretty = lev_metrics.copy()
lev_pretty["Final Value"]   = lev_pretty["Final Value"].map(lambda x: f"{x:,.0f}")
lev_pretty["Min Balance"]   = lev_pretty["Min Balance"].map(lambda x: f"{x:,.0f}")
lev_pretty["Calmar"]        = lev_pretty["Calmar"].map(lambda x: f"{x:.2f}")
lev_pretty["Sharpe"]        = lev_pretty["Sharpe"].map(lambda x: f"{x:.2f}")
lev_pretty["Abs DD vs Initial"] = lev_pretty["Abs DD vs Initial"].map(
    lambda x: f"{x*100:.2f}%"
)

for col in ["Total Return", "CAGR", "Volatility", "Max Drawdown"]:
    lev_pretty[col] = lev_pretty[col].map(lambda x: f"{x*100:.2f}%")

lev_pretty.index.name = "Strategy"
display(lev_pretty.reset_index())


Cell 15 - Professional Account

In [0]:
# --- CFD cost assumptions (approx Saxo-like) ---
COST = 0.0010                 # 10 bps per switch (spread+commission)
CFD_FIN_RATE_ANNUAL = 0.08    # 8% p.a. financing on long notional (SOFR ~5% + 3%)
CFD_FIN_RATE_HOURLY = CFD_FIN_RATE_ANNUAL / 8760.0
MAX_LEV_PMMF        = 3.0     # cap leverage on PMMF/T-bill CFD


In [0]:
# Cell 13 — Optimize BASE_LEVERAGE up to 17x (DD vs Initial ≤ 50%) with CFD financing

MIN_LEVERAGE       = 1.0
HIT_THRESHOLD      = 0.60
LEVERAGE_LOOKBACK  = 24 * 20
ABS_DD_HARD_CAP    = -0.50      # never allow equity < 50% of initial
DD_FULL_LEV        = -0.05      # DD >= -5% → full BASE_LEVERAGE
DD_NO_LEV          = -0.35      # DD <= -35% → min 1x


def run_wf_levered_with_cap(df, base_leverage: float, label: str):
    capital     = INITIAL_CAPITAL
    portfolio   = [capital]
    active      = "START"
    hourly_rets = []
    peak        = capital

    for i in range(1, len(df)):
        r = df.predicted_regime.iloc[i-1]

        # --- regime returns (unchanged) ---
        if r == 0:
            target = "QQQ"
            ret    = df.QQQ.iloc[i] / df.QQQ.iloc[i-1] - 1
        elif r == 1:
            target = "SPY"
            ret    = df.SPY.iloc[i] / df.SPY.iloc[i-1] - 1
        else:
            target = "PMMF"
            p_now  = df.PMMF.iloc[i]
            p_prev = df.PMMF.iloc[i-1]
            if pd.notna(p_now) and pd.notna(p_prev) and p_prev > 0:
                ret = p_now / p_prev - 1
            else:
                ret = MMF_HOURLY_RATE

        # update peak (diagnostic)
        if capital > peak:
            peak = capital

        # DD vs initial capital
        dd_initial = capital / INITIAL_CAPITAL - 1.0

        # --- rolling hit-rate gate ---
        if len(hourly_rets) >= LEVERAGE_LOOKBACK:
            window   = hourly_rets[-LEVERAGE_LOOKBACK:]
            hit_rate = sum(r2 >= -0.001 for r2 in window) / len(window)
            lev_hr   = base_leverage if hit_rate > HIT_THRESHOLD else 1.0
        else:
            lev_hr = 1.0

        # --- DD–scaled leverage based on dd_initial ---
        if dd_initial >= DD_FULL_LEV:
            lev_dd = base_leverage
        elif dd_initial <= DD_NO_LEV:
            lev_dd = MIN_LEVERAGE
        else:
            frac   = (dd_initial - DD_NO_LEV) / (DD_FULL_LEV - DD_NO_LEV)
            lev_dd = MIN_LEVERAGE + (base_leverage - MIN_LEVERAGE) * frac

        lev = min(lev_hr, lev_dd)

        # never lever PMMF beyond its own cap
        if target == "PMMF":
            lev = min(lev, MAX_LEV_PMMF)

        # absolute 50% DD cap vs initial
        if dd_initial <= ABS_DD_HARD_CAP and lev > 1.0:
            lev = 1.0

        # --- CFD financing cost on gross exposure > 1x ---
        gross_exposure = lev      # 1x = fully invested, lev× multiple of equity
        extra_lev      = max(gross_exposure - 1.0, 0.0)
        fin_cost       = CFD_FIN_RATE_HOURLY * extra_lev

        cost = COST if target != active else 0.0

        capital *= (1 + ret * lev - cost - fin_cost)

        portfolio.append(capital)
        hourly_rets.append(ret)
        active = target

    series = pd.Series(portfolio, index=df.index, name=label)
    return series


# run optimization for BASE_LEVERAGE from 1x to 17x
leverages_to_test = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0, 12.0, 15.0, 17.0]

optimized_series = {}
for L in leverages_to_test:
    label = f"WF + {L:.0f}x"
    optimized_series[L] = run_wf_levered_with_cap(wf_df, L, label)

# store one chosen level (e.g. 6x) into wf_df for later plotting
wf_df["WF_Baseline_Lev"] = optimized_series[6.0]
print("✅ Optimization runs with CFD costs completed for leverage levels:", leverages_to_test)


In [0]:
# Cell 14 — Metrics: Baseline + All Leverage Levels + SPY (with Abs DD vs Initial)

def min_balance_vs_initial(series):
    initial = series.iloc[0]
    min_val = series.min()
    abs_dd_initial = min_val / initial - 1.0
    return min_val, abs_dd_initial

rows = []
index = []

# baseline and SPY
series_list = [
    ("V4 WalkForward", wf_df["WF_Baseline"]),
    ("SPY",            wf_df["SPY_Value"]),
]

# add optimized leverage runs
for L, s in optimized_series.items():
    name = f"V4 WalkForward + {L:.0f}x"
    series_list.append((name, s))

for name, s in series_list:
    m = compute_metrics(s)
    min_bal, abs_dd_init = min_balance_vs_initial(s)
    m["Min Balance"]       = min_bal
    m["Abs DD vs Initial"] = abs_dd_init
    rows.append(m)
    index.append(name)

lev_metrics = pd.DataFrame(rows, index=index)

lev_pretty = lev_metrics.copy()
lev_pretty["Final Value"]   = lev_pretty["Final Value"].map(lambda x: f"{x:,.0f}")
lev_pretty["Min Balance"]   = lev_pretty["Min Balance"].map(lambda x: f"{x:,.0f}")
lev_pretty["Calmar"]        = lev_pretty["Calmar"].map(lambda x: f"{x:.2f}")
lev_pretty["Sharpe"]        = lev_pretty["Sharpe"].map(lambda x: f"{x:.2f}")
lev_pretty["Abs DD vs Initial"] = lev_pretty["Abs DD vs Initial"].map(
    lambda x: f"{x*100:.2f}%"
)

for col in ["Total Return", "CAGR", "Volatility", "Max Drawdown"]:
    lev_pretty[col] = lev_pretty[col].map(lambda x: f"{x*100:.2f}%")

lev_pretty.index.name = "Strategy"
display(lev_pretty.reset_index())


In [0]:
# ============================================================
# TRADE LOG — V4 WF Baseline (Probability Conviction Notebook)
# ============================================================
# Run after Cell 8 (wf_df has WF_Baseline, prices joined, timestamp as index)

trade_log = []
active_asset = "START"
capital      = INITIAL_CAPITAL

for i in range(1, len(wf_df)):

    regime    = wf_df["predicted_regime"].iloc[i - 1]
    timestamp = wf_df.index[i]

    if regime == 0:
        target = "QQQ"
    elif regime == 1:
        target = "SPY"
    else:
        target = "PMMF"

    # Regime switch — record the trade
    if target != active_asset:

        p_now  = wf_df[target].iloc[i]   if target != "PMMF" else wf_df["PMMF"].iloc[i]
        p_prev = wf_df[target].iloc[i-1] if target != "PMMF" else wf_df["PMMF"].iloc[i-1]

        # PMMF return with MMF accrual fallback
        if target == "PMMF":
            if pd.notna(p_now) and pd.notna(p_prev) and p_prev > 0:
                ret = p_now / p_prev - 1
            else:
                ret = MMF_HOURLY_RATE
        else:
            ret = p_now / p_prev - 1

        # Probability context at the moment of the switch
        trade_log.append({
            "timestamp"      : timestamp,
            "from_asset"     : active_asset,
            "to_asset"       : target,
            "regime"         : int(regime),
            "prob_0_qqq"     : round(wf_df["prob_0"].iloc[i-1], 4),
            "prob_1_spy"     : round(wf_df["prob_1"].iloc[i-1], 4),
            "prob_2_cash"    : round(wf_df["prob_2"].iloc[i-1], 4),
            "max_prob"       : round(max(wf_df["prob_0"].iloc[i-1],
                                         wf_df["prob_1"].iloc[i-1],
                                         wf_df["prob_2"].iloc[i-1]), 4),
            "conviction"     : "HIGH" if max(wf_df["prob_0"].iloc[i-1],
                                              wf_df["prob_1"].iloc[i-1],
                                              wf_df["prob_2"].iloc[i-1]) >= CONVICTION_THRESHOLD else "LOW",
            "price_entry"    : round(p_now, 4) if pd.notna(p_now) else None,
            "transaction_cost": COST,
            "portfolio_value": round(wf_df["WF_Baseline"].iloc[i], 2),
        })

        capital *= (1 + ret - COST)
    else:
        # No switch — still apply return
        p_now  = wf_df[target].iloc[i]
        p_prev = wf_df[target].iloc[i-1]
        if target == "PMMF":
            ret = (p_now / p_prev - 1) if (pd.notna(p_now) and pd.notna(p_prev) and p_prev > 0) else MMF_HOURLY_RATE
        else:
            ret = p_now / p_prev - 1
        capital *= (1 + ret)

    active_asset = target

trades_df = pd.DataFrame(trade_log)
trades_df["timestamp"] = pd.to_datetime(trades_df["timestamp"])

print(f"Total trades: {len(trades_df)}")
print(f"\nBreakdown by destination:\n{trades_df['to_asset'].value_counts()}")
print(f"\nConviction split:\n{trades_df['conviction'].value_counts()}")
display(trades_df)

In [0]:
# ============================================================
# V4 SHORT STRATEGY — FIXED FOR WF NOTEBOOK
# Uses wf_df (NOT df)
# ============================================================

INITIAL_CAPITAL = 5_000_000
TRADING_DAYS    = 365

SWITCH_COST = 0.001
BASE_COST   = 0.0005
BORROW_COST = 0.02 / 365


# ─────────────────────────────────────────────
# BASELINE (matches your WF baseline logic)
# ─────────────────────────────────────────────
def run_v4_baseline(data):
    capital = INITIAL_CAPITAL
    portfolio = [capital]
    active = "START"

    for i in range(1, len(data)):
        r = data.predicted_regime.iloc[i-1]

        if r == 0:
            target = "QQQ"
            ret = data.QQQ.iloc[i] / data.QQQ.iloc[i-1] - 1

        elif r == 1:
            target = "SPY"
            ret = data.SPY.iloc[i] / data.SPY.iloc[i-1] - 1

        else:
            target = "CASH"
            ret = 0.0

        cost = BASE_COST if target != active else 0
        capital *= (1 + ret - cost)

        portfolio.append(capital)
        active = target

    return portfolio


# ─────────────────────────────────────────────
# SHORT VERSION
# ─────────────────────────────────────────────
def run_v4_with_short(data):
    capital = INITIAL_CAPITAL
    portfolio = [capital]
    active = "START"

    for i in range(1, len(data)):
        r = data.predicted_regime.iloc[i-1]

        # ─────────────────────────────
        # EXPANSION → QQQ
        # ─────────────────────────────
        if r == 0:
            target = "QQQ"
            ret = data.QQQ.iloc[i] / data.QQQ.iloc[i-1] - 1

        # ─────────────────────────────
        # SLOWDOWN → SPY
        # ─────────────────────────────
        elif r == 1:
            target = "SPY"
            ret = data.SPY.iloc[i] / data.SPY.iloc[i-1] - 1

        # ─────────────────────────────
        # CONTRACTION → CONDITIONAL SHORT
        # ─────────────────────────────
        elif r == 2:
            p2  = data.prob_2.iloc[i-1]
            mom = data.Price_Mom_24H.iloc[i-1]

            # --- probability-based sizing ---
            if p2 > 0.7:
                weight = 0.5
            elif p2 > 0.6:
                weight = 0.25
            else:
                weight = 0.0

            # --- momentum filter (KEY FIX) ---
            if (weight > 0) and (mom < 0):
                target = "SHORT_SPY"
                ret_spy = data.SPY.iloc[i] / data.SPY.iloc[i-1] - 1
                ret = -weight * ret_spy - weight * BORROW_COST
            else:
                target = "CASH"
                ret = 0.0

        # ─────────────────────────────
        # SAFETY FALLBACK
        # ─────────────────────────────
        else:
            target = "CASH"
            ret = 0.0

        # ─────────────────────────────
        # TRANSACTION COST
        # ─────────────────────────────
        cost = SWITCH_COST if target != active else 0.0

        capital *= (1 + ret - cost)
        portfolio.append(capital)
        active = target

    return portfolio


# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────
wf_df["V4_Base"]  = run_v4_baseline(wf_df)
wf_df["V4_Short"] = run_v4_with_short(wf_df)

wf_df["SPY_Value"] = (
    wf_df["SPY"] / wf_df["SPY"].iloc[0]
) * INITIAL_CAPITAL


# ─────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────
def compute_metrics(series):
    r = series.pct_change().dropna()

    years = len(r) / 8760  # hourly

    total_return = series.iloc[-1] / series.iloc[0] - 1
    cagr = (series.iloc[-1] / series.iloc[0]) ** (1 / years) - 1
    vol = r.std() * np.sqrt(8760)
    sharpe = r.mean() / r.std() * np.sqrt(8760)

    dd = series / series.cummax() - 1
    maxdd = dd.min()

    return {
        "Total Return": total_return,
        "CAGR": cagr,
        "Volatility": vol,
        "Sharpe": sharpe,
        "Max DD": maxdd
    }


perf = pd.DataFrame([
    compute_metrics(wf_df["V4_Base"]),
    compute_metrics(wf_df["V4_Short"]),
    compute_metrics(wf_df["SPY_Value"]),
], index=["V4 Base", "V4 + Short", "SPY"])


# ─────────────────────────────────────────────
# DISPLAY
# ─────────────────────────────────────────────
fmt = perf.copy()
for col in ["Total Return", "CAGR", "Volatility", "Max DD"]:
    fmt[col] = fmt[col].map(lambda x: f"{x*100:.2f}%")
fmt["Sharpe"] = fmt["Sharpe"].map(lambda x: f"{x:.2f}")

print("\n=== PERFORMANCE ===")
print(fmt)


# ─────────────────────────────────────────────
# EXPOSURE CHECK
# ─────────────────────────────────────────────
short_ratio = (wf_df["predicted_regime"] == 2).mean()
print(f"\nShort Exposure: {short_ratio:.2%}")


# ─────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────
plt.figure(figsize=(16,7))

plt.plot(wf_df["V4_Base"], label="V4 Base")
plt.plot(wf_df["V4_Short"], label="V4 + Short")
plt.plot(wf_df["SPY_Value"], label="SPY")

plt.yscale("log")
plt.legend()
plt.grid(alpha=0.3)
plt.title("V4 vs V4 + Short")

plt.show()

# ============================================================
high_conf_short = ((wf_df["predicted_regime"] == 2) & (wf_df["prob_2"] > 0.6)).mean()
print(f"High-confidence short exposure: {high_conf_short:.2%}")